In [1]:
import itertools
import os
import time
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
from matplotlib import rcParams
from any_functions import create_data_folder
from write_arrays_to_excel import write_arrays_excel
from create_map_and_save import create_map_and_save


# импорт настроек
from config import *
# ===================
# импорт драйверов
# ===================

from devices.yokogawa.OSA_Yokogawa_new import OSA_Yokogawa_new
from devices.btf_100.btf_100 import BTF100
from devices.rsa_device.rf_measure_lib import rf_measurement
from devices.rsa_device.RF306B import RF306B
from devices.odl_650.OpticDelayLine_new import OpticDelayLine
from devices.ut_3005.VoltageDriverUT3005 import VoltageDriverUT3005
from devices.cdl_1015.CLD1015 import CLD1015




# ============================================================
# Функция для построение 6 карт для одного напряжения
# ============================================================


def _build_voltage_maps(voltage, main_folder, data_buf):
    """Внутренняя функция: строит 7 тепловых карт (6 RF и мощемер) для заданного напряжения."""
    from pathlib import Path
    import numpy as np
    
    # Фильтруем данные по напряжению
    mask = np.array(data_buf['voltage']) == voltage
    
    # Папка для карт этого напряжения
    plot_subfolder = Path(main_folder) / f"voltage_{voltage}V" / "maps"
    plot_subfolder.mkdir(parents=True, exist_ok=True)
    
    # Данные для осей
    x = np.array(data_buf['current'])[mask]
    y = np.array(data_buf['delay'])[mask]
    
    maps_config = [
        (data_buf['rf_freq_max'], 'rf_freq_max', 'Частота, ГГц (max span)'),
        (data_buf['rf_pow_max'],  'rf_pow_max',  'Мощность, дБм (max span)'),
        (data_buf['rf_freq_mid'], 'rf_freq_mid', 'Частота, ГГц (mid span)'),
        (data_buf['rf_pow_mid'],  'rf_pow_mid',  'Мощность, дБм (mid span)'),
        (data_buf['rf_freq_min'], 'rf_freq_min', 'Частота, ГГц (min span)'),
        (data_buf['rf_pow_min'],  'rf_pow_min',  'Мощность, дБм (min span)'),
    ]
    
    for z_raw, fname_suffix, z_label in maps_config:
        z = np.array(z_raw)[mask]
        create_map_and_save(
            x_arr=[x.tolist(), "Ток, мА"],
            y_arr=[y.tolist(), "Задержка, пс"],
            z_arr=[z.tolist(), z_label],
            title=f"U = {voltage}В, {z_label}",
            folder_path=plot_subfolder,
            filename=f"RF_{fname_suffix}",
            show_plot=False  # 👈 True только для отладки
        )
    


def main():
    main_folder = create_data_folder(prefix='laser_with_BTF')
    
    params = itertools.product(VOLTAGES, CURRENTS, DELAYS)

    # === Буфер для сбора данных ===
    collected_data = {
        'voltage': [], 'current': [], 'delay': [],
        'rf_freq_max': [], 'rf_pow_max': [],
        'rf_freq_mid': [], 'rf_pow_mid': [],
        'rf_freq_min': [], 'rf_pow_min': [],
    }
    
    current_voltage_batch = None

    # === Списки для Excel ===
    idx_data, time_data, voltage_data, delay_data = [], [], [], []
    current_data = []
    rf_freq_max_data, rf_pow_max_data = [], []
    rf_freq_mid_data, rf_pow_mid_data = [], []
    rf_freq_min_data, rf_pow_min_data = [], []
    
    voltage_prev=-1
    current_prev=-1
    delay_prev=-1
    
    try:
        # ========================
        # ИНИЦИАЛИЗАЦИЯ ПРИБОРОВ``
        # ========================
        odl = OpticDelayLine('COM10')
        odl.initialize()
        # osa = OSA_Yokogawa_new()
        rf_device = RF306B()
        LD = CLD1015()
        LD.turn_on_all()

        ut = VoltageDriverUT3005('COM8')
        ut.turn_on()

        for idx, (voltage, current, delay) in enumerate(params, 1):

            # === Построение карт при смене напряжения ===
            if current_voltage_batch is not None and voltage != current_voltage_batch:
                _build_voltage_maps(current_voltage_batch, main_folder, collected_data)
            current_voltage_batch = voltage

            
            rf_folder_name = (f"voltage_{voltage}V/rf_measurements/current_{current}mA")
            rf_full_path = os.path.join(main_folder, rf_folder_name)

            # === НАСТРОЙКА ОБОРУДОВАНИЯ ===
            
            # === НАСТРОЙКА ОБОРУДОВАНИЯ ===
            voltage_next=voltage
            if voltage_prev!=voltage_next:
                ut.set_voltage(voltage=voltage)
                voltage_prev=voltage_next
            
            current_next=current
            if current_prev!=current_next:
                LD.set_current(current=current)
                current_prev=current_next
            
            delay_next=delay
            if delay_prev!=delay_next:
                odl.set_time_delay(time_delay=delay)
                delay_prev=delay_next
            
            time.sleep(3)
            

            # === ИЗМЕРЕНИЯ ===
            
            

            base_filename=f"delay_{delay}_"
            

            rf_freq_max, rf_pow_max = rf_measurement(
                rf_device=rf_device, N=NUMBER_RF_MEASURE, folder_path=rf_full_path,
                filename=base_filename+'rf_max_span', rf_rbw=RF_RBW_MAX,
                f_start=RF_F_START_MAX, f_stop=RF_F_STOP_MAX, save_png=True)

            
            rf_freq_mid, rf_pow_mid = rf_measurement(
                rf_device=rf_device, N=NUMBER_RF_MEASURE, folder_path=rf_full_path,
                filename=base_filename+'rf_middle_span', rf_rbw=RF_RBW_MIDDLE,f_span=RF_SPAN_MIDDLE,f_center=rf_freq_max, save_png=True)

            
            rf_freq_min, rf_pow_min = rf_measurement(
                rf_device=rf_device, N=NUMBER_RF_MEASURE, folder_path=rf_full_path,
                filename=base_filename+'rf_min_span', rf_rbw=RF_RBW_MIN,
            f_center=rf_freq_mid, f_span=RF_SPAN_MIN,save_png=True)


            # === НАКОПЛЕНИЕ ДАННЫХ ===
            idx_data.append(idx)
            time_data.append(datetime.now())
            voltage_data.append(voltage)
            delay_data.append(delay)
            current_data.append(current)
            rf_freq_max_data.append(rf_freq_max)
            rf_pow_max_data.append(rf_pow_max)
            rf_freq_mid_data.append(rf_freq_mid)
            rf_pow_mid_data.append(rf_pow_mid)
            rf_freq_min_data.append(rf_freq_min)
            rf_pow_min_data.append(rf_pow_min)

            # === Заполнение буфера для карт ===
            collected_data['voltage'].append(voltage)
            collected_data['current'].append(current)
            collected_data['delay'].append(delay)
            collected_data['rf_freq_max'].append(rf_freq_max)
            collected_data['rf_pow_max'].append(rf_pow_max)
            collected_data['rf_freq_mid'].append(rf_freq_mid)
            collected_data['rf_pow_mid'].append(rf_pow_mid)
            collected_data['rf_freq_min'].append(rf_freq_min)
            collected_data['rf_pow_min'].append(rf_pow_min)
            

            

        # === Построение карт для ПОСЛЕДНЕГО напряжения ===
        if current_voltage_batch is not None:
            _build_voltage_maps(current_voltage_batch, main_folder, collected_data)

        
        idx_arr = [idx_data, 'number']
        time_arr = [time_data, 'time']
        voltage_arr = [voltage_data, 'voltage_V']
        delay_arr = [delay_data, 'delay_ps']
        current_arr = [current_data, 'current_mA']
        rf_freq_max_arr = [rf_freq_max_data, 'rf_peak_freq_max_GHz']
        rf_pow_max_arr = [rf_pow_max_data, 'rf_peak_power_max_dBm']
        rf_freq_mid_arr = [rf_freq_mid_data, 'rf_peak_freq_middle_GHz']
        rf_pow_mid_arr = [rf_pow_mid_data, 'rf_peak_power_middle_dBm']
        rf_freq_min_arr = [rf_freq_min_data, 'rf_peak_freq_min_GHz']
        rf_pow_min_arr = [rf_pow_min_data, 'rf_peak_power_min_dBm']

        write_arrays_excel(
            idx_arr, time_arr, voltage_arr, delay_arr,
            rf_freq_max_arr, rf_pow_max_arr,
            rf_freq_mid_arr, rf_pow_mid_arr,
            rf_freq_min_arr, rf_pow_min_arr,
            folder_path=main_folder, filename='results.xlsx')  
    finally:
        try:
            odl.disconnect()
            LD.turn_off_all()
            ut.out_off_and_close_COM()
        except Exception as e:
            print(f"Ошибка при отключении: {e}")    
        
    

if __name__ == "__main__":
    main()


✅ Порт COM10 открыт
API Version 3.11.0047.0
One device found.
Device type: RSA306B
Device serial number: B032586
RF306B inited!
CLD1015 inited!
tec is ON
laser is ON
Voltage Driver is connected
The output is on
Set voltage =  0
Set laser current =  0.1
🎯 Установка задержки: 0
➡️  Отправлено: T0
⬅️  Получено: T0
⚠️  Ответ не 'Done'. Попытка 1 из 10
💤 Пауза 1.0 сек перед повторной отправкой...
➡️  Отправлено: T0
⬅️  Получено: T0
⬅️  Получено: Done
✅ Задержка установлена успешно (попытка 2)
Данные успешно сохранены
Данные успешно сохранены
Данные успешно сохранены
🎯 Установка задержки: 20
➡️  Отправлено: T20
⬅️  Получено: T20
⬅️  Получено: Done
✅ Задержка установлена успешно (попытка 1)
Данные успешно сохранены
Данные успешно сохранены
Данные успешно сохранены
🎯 Установка задержки: 40
➡️  Отправлено: T40
⬅️  Получено: T40
⚠️  Ответ не 'Done'. Попытка 1 из 10
💤 Пауза 1.0 сек перед повторной отправкой...
➡️  Отправлено: T40
⬅️  Получено: T40
⬅️  Получено: Done
✅ Задержка установлена успешно

In [ ]:
import numpy as np
DELAYS=np.linspace(0, 300, 11)
print(DELAYS)

In [8]:
time_arr=[1,2]
voltage_arr=[3,4]
# del time_arr, voltage_arr

In [11]:
print(time_arr)

NameError: name 'time_arr' is not defined

In [10]:
del time_arr